The FDA runs a database called FAERS. It collects reports of harmful reactions people have after taking approved drugs, called adverse events. Every quarter the FDA releases a fresh batch of reports as raw text files, and this notebook works through one quarter, 2024 Q1.

The goal is simple. Load the files exactly as they arrive. Confirm the tables connect the way the FAERS docs describe. Then flag the messy parts that I will clean later, like drug names that never match and cases that appear more than once.

Nothing gets typed or cleaned yet. Every column comes in as text, so the raw data stays true to the source. The first cell installs Spark, mounts the Drive, and loads the shared config.

In [ ]:
%pip install pyspark==3.5.0 delta-spark==3.2.0 requests -q

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = "/content/drive/MyDrive/faers-data-pipeline"
sys.path.insert(0, PROJECT_ROOT)

with open(os.path.join(PROJECT_ROOT, "config/pipeline_config.json")) as f:
    CONFIG = json.load(f)

print(f"✓ Project root: {PROJECT_ROOT}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
Mounted at /content/drive
✓ Project root: /content/drive/MyDrive/faers-data-pipeline


We start by seeing what actually landed in the folder. Then it reads a single raw line. The files are plain text, and a dollar sign separates the columns instead of a comma. A quick peek confirms the separator, shows the column names, and proves the files decoded without garbled characters.

In [ ]:
raw_dir = os.path.join(PROJECT_ROOT, "data/raw/2024q1")
print(f"Looking in: {raw_dir}\n")

if os.path.exists(raw_dir):
    for f in sorted(os.listdir(raw_dir)):
        size_mb = os.path.getsize(os.path.join(raw_dir, f)) / (1024 * 1024)
        print(f"  {f:<20} {size_mb:>8.1f} MB")
else:
    print("⚠ Folder not found! Upload your TXT files to:")
    print(f"  Google Drive > faers-data-pipeline > data > raw > 2024q1")

Looking in: /content/drive/MyDrive/faers-data-pipeline/data/raw/2024q1

  DEMO24Q1.txt             57.4 MB
  DRUG24Q1.txt            186.8 MB
  INDI24Q1.txt             55.4 MB
  OUTC24Q1.txt              6.2 MB
  REAC24Q1.txt             53.4 MB
  RPSR24Q1.txt              0.3 MB
  THER24Q1.txt             20.6 MB


In [ ]:
raw_dir = os.path.join(PROJECT_ROOT, "data/raw/2024q1")

for filename in ["DEMO24Q1.txt", "DRUG24Q1.txt", "REAC24Q1.txt"]:
    filepath = os.path.join(raw_dir, filename)
    if not os.path.exists(filepath):
        # Try lowercase
        filepath = os.path.join(raw_dir, filename.lower())
    if not os.path.exists(filepath):
        print(f"⚠ {filename} not found, skipping\n")
        continue

    print(f"{'='*60}")
    print(f"FILE: {filename}")
    print(f"{'='*60}")

    with open(filepath, "r", encoding="ISO-8859-1") as f:
        lines = [f.readline() for _ in range(4)]  # header + 3 rows

    # Header (column names)
    header = lines[0].strip().split("$")
    print(f"Columns ({len(header)}): {header}\n")

    # Show first 3 data rows
    for i, line in enumerate(lines[1:], 1):
        fields = line.strip().split("$")
        print(f"Row {i} ({len(fields)} fields):")
        for col, val in zip(header, fields):
            if val:  # only show fields that have a value
                print(f"  {col}: {val[:60]}{'...' if len(val) > 60 else ''}")
        print()

FILE: DEMO24Q1.txt
Columns (25): ['primaryid', 'caseid', 'caseversion', 'i_f_code', 'event_dt', 'mfr_dt', 'init_fda_dt', 'fda_dt', 'rept_cod', 'auth_num', 'mfr_num', 'mfr_sndr', 'lit_ref', 'age', 'age_cod', 'age_grp', 'sex', 'e_sub', 'wt', 'wt_cod', 'rept_dt', 'to_mfr', 'occp_cod', 'reporter_country', 'occr_country']

Row 1 (25 fields):
  primaryid: 1001678125
  caseid: 10016781
  caseversion: 25
  i_f_code: F
  event_dt: 20120330
  mfr_dt: 20240125
  init_fda_dt: 20140318
  fda_dt: 20240129
  rept_cod: EXP
  mfr_num: PHHY2012CA028320
  mfr_sndr: NOVARTIS
  age: 56
  age_cod: YR
  sex: F
  e_sub: Y
  rept_dt: 20240129
  occp_cod: HP
  reporter_country: CA
  occr_country: CA

Row 2 (25 fields):
  primaryid: 1002872124
  caseid: 10028721
  caseversion: 24
  i_f_code: F
  event_dt: 20171025
  mfr_dt: 20240228
  init_fda_dt: 20140321
  fda_dt: 20240304
  rept_cod: EXP
  mfr_num: PHHY2014CA033801
  mfr_sndr: NOVARTIS
  age: 57
  age_cod: YR
  sex: F
  e_sub: Y
  rept_dt: 20240304
  occp_cod

In [ ]:
from src.utils.spark_session import get_spark

spark = get_spark("FAERS-Explore")

raw_dir = os.path.join(PROJECT_ROOT, "data/raw/2024q1")

def read_faers_file(spark, raw_dir, file_pattern):
    """Read one FAERS file into Spark. The dollar sign splits the columns."""
    # Try common name patterns
    import glob
    matches = glob.glob(os.path.join(raw_dir, file_pattern))
    if not matches:
        print(f"⚠ No files matching {file_pattern} in {raw_dir}")
        return None
    path = matches[0]
    print(f"Reading: {os.path.basename(path)}")
    return spark.read.csv(
        path,
        sep="$",
        header=True,
        inferSchema=False,       # Everything as StringType in Bronze
        encoding="ISO-8859-1",   # Handle international characters
        multiLine=True,          # Handle embedded newlines in drug names
        mode="PERMISSIVE",
    )

demo_df = read_faers_file(spark, raw_dir, "*EMO*")
drug_df = read_faers_file(spark, raw_dir, "*RUG*")
reac_df = read_faers_file(spark, raw_dir, "*EAC*")

print(f"\nRow counts:")
print(f"  DEMO: {demo_df.count():,} reports")
print(f"  DRUG: {drug_df.count():,} drug records")
print(f"  REAC: {reac_df.count():,} reaction records")

Reading: DEMO24Q1.txt
Reading: DRUG24Q1.txt
Reading: REAC24Q1.txt

Row counts:
  DEMO: 406,184 reports
  DRUG: 1,909,327 drug records
  REAC: 1,445,416 reaction records


DEMO, DRUG, and REAC carry most of a FAERS quarter. DEMO gives you one row per report. DRUG and REAC give you one or more rows per report, because a single report can name several drugs and several reactions. All three point back to a report through primaryid. That shared key is what lets you join them later.

In [ ]:
print("DEMO schema:")
demo_df.printSchema()

print("\nSample DEMO rows:")
demo_df.select(
    "primaryid", "caseid", "caseversion", "event_dt",
    "age", "age_cod", "sex", "reporter_country", "occr_country"
).limit(10).toPandas()

DEMO schema:
root
 |-- primaryid: string (nullable = true)
 |-- caseid: string (nullable = true)
 |-- caseversion: string (nullable = true)
 |-- i_f_code: string (nullable = true)
 |-- event_dt: string (nullable = true)
 |-- mfr_dt: string (nullable = true)
 |-- init_fda_dt: string (nullable = true)
 |-- fda_dt: string (nullable = true)
 |-- rept_cod: string (nullable = true)
 |-- auth_num: string (nullable = true)
 |-- mfr_num: string (nullable = true)
 |-- mfr_sndr: string (nullable = true)
 |-- lit_ref: string (nullable = true)
 |-- age: string (nullable = true)
 |-- age_cod: string (nullable = true)
 |-- age_grp: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- e_sub: string (nullable = true)
 |-- wt: string (nullable = true)
 |-- wt_cod: string (nullable = true)
 |-- rept_dt: string (nullable = true)
 |-- to_mfr: string (nullable = true)
 |-- occp_cod: string (nullable = true)
 |-- reporter_country: string (nullable = true)
 |-- occr_country: string (nullable = tru

,primaryid,caseid,caseversion,event_dt,age,age_cod,sex,reporter_country,occr_country
0,1001678125,10016781,25,20120330,56,YR,F,CA,CA
1,1002872124,10028721,24,20171025,57,YR,F,CA,CA
2,100293663,10029366,3,None,32,YR,M,AU,AU
3,1005450710,10054507,10,None,68,YR,F,US,US
4,1005762118,10057621,18,20140204,57,YR,M,CA,CA
5,1007468611,10074686,11,20090714,52,YR,M,CA,CA
6,1013616272,10136162,72,20120821,39,YR,F,CA,CA
7,1014222252,10142222,52,20131201,52,YR,M,CA,None
8,1014280316,10142803,16,20130101,57,YR,M,CA,CA
9,101539902,10153990,2,20140301,44,YR,F,US,US


In [ ]:
from pyspark.sql.functions import col, upper

print("DRUG schema:")
drug_df.printSchema()

# Drug role distribution. PS means Primary Suspect, the role that matters most.
print("\nDrug role distribution:")
drug_df.groupBy("role_cod").count().orderBy(col("count").desc()).show()

# How many unique drug names?
unique_drugs = drug_df.select("drugname").distinct().count()
print(f"Unique drug names: {unique_drugs:,}")

# Show the drug name chaos. One drug shows up under many different names.
print("\nExample: variations of atorvastatin (Lipitor):")
drug_df.filter(
    upper(col("drugname")).contains("ATORVA") |
    upper(col("drugname")).contains("LIPITOR")
).select("drugname", "prod_ai", "role_cod").distinct().show(20, truncate=False)

print("\nSample suspect drugs (role_cod = PS):")
drug_df.filter(col("role_cod") == "PS").select(
    "primaryid", "drugname", "prod_ai", "route", "dose_amt", "dose_unit"
).limit(10).toPandas()

DRUG schema:
root
 |-- primaryid: string (nullable = true)
 |-- caseid: string (nullable = true)
 |-- drug_seq: string (nullable = true)
 |-- role_cod: string (nullable = true)
 |-- drugname: string (nullable = true)
 |-- prod_ai: string (nullable = true)
 |-- val_vbm: string (nullable = true)
 |-- route: string (nullable = true)
 |-- dose_vbm: string (nullable = true)
 |-- cum_dose_chr: string (nullable = true)
 |-- cum_dose_unit: string (nullable = true)
 |-- dechal: string (nullable = true)
 |-- rechal: string (nullable = true)
 |-- lot_num: string (nullable = true)
 |-- exp_dt: string (nullable = true)
 |-- nda_num: string (nullable = true)
 |-- dose_amt: string (nullable = true)
 |-- dose_unit: string (nullable = true)
 |-- dose_form: string (nullable = true)
 |-- dose_freq: string (nullable = true)


Drug role distribution:
+--------+------+
|role_cod| count|
+--------+------+
|      SS|819690|
|       C|672903|
|      PS|406273|
|       I| 10461|
+--------+------+

Unique drug n

,primaryid,drugname,prod_ai,route,dose_amt,dose_unit
0,1001678125,SANDOSTATIN,OCTREOTIDE ACETATE,Subcutaneous,None,None
1,1002872124,SANDOSTATIN,OCTREOTIDE ACETATE,Subcutaneous,None,None
2,100293663,CYCLOSPORINE,CYCLOSPORINE,Unknown,None,None
3,1005450710,ENBREL,ETANERCEPT,Unknown,None,None
4,1005762118,XOLAIR,OMALIZUMAB,Subcutaneous,225,MG
5,1007468611,XOLAIR,OMALIZUMAB,Subcutaneous,225,MG
6,1013616272,XOLAIR,OMALIZUMAB,Subcutaneous,300,MG
7,1014222252,ACTEMRA,TOCILIZUMAB,Intravenous (not otherwise specified),350,MG
8,1014280316,XOLAIR,OMALIZUMAB,Subcutaneous,225,MG
9,101539902,MESALAMINE,MESALAMINE,Oral,3.6,GM


In [ ]:
from pyspark.sql.functions import col

print("REAC schema:")
reac_df.printSchema()

unique_terms = reac_df.select("pt").distinct().count()
print(f"\nUnique MedDRA Preferred Terms: {unique_terms:,}")

print("\nTop 20 most reported adverse reactions:")
reac_df.groupBy("pt").count().orderBy(col("count").desc()).limit(20).toPandas()

REAC schema:
root
 |-- primaryid: string (nullable = true)
 |-- caseid: string (nullable = true)
 |-- pt: string (nullable = true)
 |-- drug_rec_act: string (nullable = true)


Unique MedDRA Preferred Terms: 12,361

Top 20 most reported adverse reactions:


,pt,count
0,Off label use,34928
1,Drug ineffective,24019
2,Fatigue,20713
3,Diarrhoea,17537
4,Nausea,16848
5,Product dose omission issue,16638
6,Death,15907
7,Headache,13236
8,COVID-19,13200
9,Pain,12976


Here is a quick check that the tables really connect. It joins the primary suspect drugs to their reactions on primaryid. 

In [ ]:
from pyspark.sql.functions import col

# Join suspect drugs to their reactions
suspect_drugs = drug_df.filter(col("role_cod") == "PS") \
    .select("primaryid", "drugname")

drug_reactions = suspect_drugs.join(reac_df.select("primaryid", "pt"),
                                    on="primaryid")

print(f"Drug-reaction pairs (suspect drugs only): {drug_reactions.count():,}")

# Show a sample
print("\nSample drug-reaction pairs:")
drug_reactions.limit(15).toPandas()

Drug-reaction pairs (suspect drugs only): 1,445,597

Sample drug-reaction pairs:


,primaryid,drugname,pt
0,100293663,CYCLOSPORINE,Toxicity to various agents
1,100293663,CYCLOSPORINE,Mycobacterium haemophilum infection
2,100293663,CYCLOSPORINE,Wound infection staphylococcal
3,100293663,CYCLOSPORINE,Staphylococcal infection
4,1005450710,ENBREL,Drug eruption
5,1005450710,ENBREL,Drug hypersensitivity
6,1007468611,XOLAIR,Middle insomnia
7,1007468611,XOLAIR,Wheezing
8,1007468611,XOLAIR,Productive cough
9,1007468611,XOLAIR,Restrictive pulmonary disease


This checks the null rates and a few distributions. FAERS reports often lack dates and ages, so some blanks are normal. What matters is knowing how much is missing, so the later code can plan for it instead of breaking on it.

In [ ]:
from pyspark.sql.functions import col

print("=" * 60)
print("DATA QUALITY SNAPSHOT — 2024 Q1")
print("=" * 60)

total = demo_df.count()
print(f"\nTotal reports: {total:,}\n")

# Null rates on key DEMO fields
print("DEMO null rates:")
for c in ["primaryid", "caseid", "event_dt", "age", "sex",
          "reporter_country", "occr_country"]:
    nulls = demo_df.filter(col(c).isNull() | (col(c) == "")).count()
    pct = nulls / total * 100
    print(f"  {c:<20} {nulls:>8,} missing ({pct:.1f}%)")

# Sex distribution
print("\nSex distribution:")
demo_df.groupBy("sex").count().orderBy(col("count").desc()).show()

# Top 10 reporting countries
print("Top 10 reporting countries:")
demo_df.groupBy("reporter_country").count() \
    .orderBy(col("count").desc()).limit(10).toPandas()

DATA QUALITY SNAPSHOT — 2024 Q1

Total reports: 406,184

DEMO null rates:
  primaryid                   0 missing (0.0%)
  caseid                      0 missing (0.0%)
  event_dt              222,393 missing (54.8%)
  age                   157,772 missing (38.8%)
  sex                    62,544 missing (15.4%)
  reporter_country            0 missing (0.0%)
  occr_country           37,679 missing (9.3%)

Sex distribution:
+----+------+
| sex| count|
+----+------+
|   F|204178|
|   M|137899|
|NULL| 62544|
| UNK|  1563|
+----+------+

Top 10 reporting countries:


,reporter_country,count
0,US,269734
1,CA,29537
2,JP,15919
3,GB,13984
4,FR,13918
5,DE,8313
6,CN,6586
7,IT,5468
8,ES,4265
9,AU,3598


FAERS lets people resubmit a report as new details arrive so the same caseid can show up several times, each with a higher caseversion. Only the top version counts. This cell measures how common that is.

In [ ]:
from pyspark.sql.functions import col, countDistinct, max as spark_max

# How many caseids have multiple versions?
case_versions = demo_df.groupBy("caseid").agg(
    countDistinct("caseversion").alias("num_versions"),
    spark_max("caseversion").alias("max_version")
)

multi_version = case_versions.filter(col("num_versions") > 1)
print(f"Cases with multiple versions: {multi_version.count():,} "
      f"out of {case_versions.count():,} unique caseids")
print(f"(These need deduplication — keep only the highest caseversion)\n")

print("Examples of multi-version cases:")
multi_version.orderBy(col("num_versions").desc()).limit(10).toPandas()

Cases with multiple versions: 0 out of 406,184 unique caseids
(These need deduplication — keep only the highest caseversion)

Examples of multi-version cases:


,caseid,num_versions,max_version


Each quarter ships more than the core three. OUTC records outcomes like a hospital stay or death. RPSR tracks who filed the report. THER holds the therapy dates. INDI gives the indication, meaning the reason someone took the drug. 

In [ ]:
outc_df = read_faers_file(spark, raw_dir, "*UTC*") or read_faers_file(spark, raw_dir, "*utc*")
rpsr_df = read_faers_file(spark, raw_dir, "*PSR*") or read_faers_file(spark, raw_dir, "*psr*")
ther_df = read_faers_file(spark, raw_dir, "*HER*") or read_faers_file(spark, raw_dir, "*her*")
indi_df = read_faers_file(spark, raw_dir, "*NDI*") or read_faers_file(spark, raw_dir, "*ndi*")

if outc_df:
    print(f"\nOUTC rows: {outc_df.count():,}")
    print("Outcome distribution:")
    outc_df.groupBy("outc_cod").count().orderBy(col("count").desc()).show()

if indi_df:
    print(f"\nINDI rows: {indi_df.count():,}")
    print("Top 10 indications (why the drug was prescribed):")
    indi_df.groupBy("indi_pt").count().orderBy(col("count").desc()).limit(10).toPandas()

Reading: OUTC24Q1.txt
Reading: RPSR24Q1.txt
Reading: THER24Q1.txt
Reading: INDI24Q1.txt

OUTC rows: 295,044
Outcome distribution:
+--------+------+
|outc_cod| count|
+--------+------+
|      OT|160229|
|      HO| 82534|
|      DE| 32303|
|      LT| 11990|
|      DS|  6112|
|      RI|   981|
|      CA|   895|
+--------+------+


INDI rows: 1,186,115
Top 10 indications (why the drug was prescribed):


The full quarter is far too big to commit to a repo. So this pulls a small part of the data to put into the repo.

In [ ]:
import csv

sample_dir = os.path.join(PROJECT_ROOT, "data/sample")
os.makedirs(sample_dir, exist_ok=True)

# Get 200 primaryids from DEMO to use as a consistent sample
sample_ids = [row.primaryid for row in demo_df.limit(200).select("primaryid").collect()]

for name, df in [("DEMO_SAMPLE.txt", demo_df),
                 ("DRUG_SAMPLE.txt", drug_df),
                 ("REAC_SAMPLE.txt", reac_df)]:
    sample = df.filter(col("primaryid").isin(sample_ids))
    sample_path = os.path.join(sample_dir, name)

    # Write with the dollar sign separator to match the original files.
    pdf = sample.toPandas()
    pdf.to_csv(sample_path, sep="$", index=False)

    print(f"✓ {name}: {len(pdf)} rows, "
          f"{os.path.getsize(sample_path) / 1024:.0f} KB")

print(f"\nSample files written to: data/sample/")

✓ DEMO_SAMPLE.txt: 200 rows, 33 KB
✓ DRUG_SAMPLE.txt: 3849 rows, 390 KB
✓ REAC_SAMPLE.txt: 3089 rows, 114 KB

Sample files written to: data/sample/
